# xQTL AD loci table

Reproducible pipeline to generate the unified AD xQTL loci summary Excel table and deploy the Shiny explorer.

You will need:
1. Metadata
- `columns_metadata.tsv` containing all columns to keep for the excel sheets, with associated metadata (column width, coloring..)
- `excel_metadata.tsv` containing some meta information for the excel sheet construction
- `pattern_coloring.tsv` containing all pattern to color in the excel sheet

2. Flatten tables
- `xQTL_all_methods_overlap_with_AD_loci_unified_cs95orColocs_Pval1e5_202604.csv.gz` — one row per variant × method × context × gene combination at AD loci
- `AD_loci_unified_cs95orColocs_Pval1e5_variant_level.csv.gz` — one row per variant at AD loci with GWAS-level annotations

3. Alexandre's function
-  `gene_prio_utils.R`

The updated data can be downloaded from: `s3://statfungen/ftp_fgc_xqtl/interactive_analysis/JennyE/xQTL_AD_loci_table/`


## Libraries

In [10]:
library(data.table)
library(openxlsx)
library(stringr)

## Set up working directory

In [11]:
out <- '/restricted/projectnb/xqtl/jaempawi/xqtl/AD_xQTL_loci/'
# out <- '/data/interactive_analysis/JennyE/xQTL_AD_loci_table/'
source(file.path(out, 'gene_prio_utils.R'))

1 threads available for data.table

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.0     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ tibble    3.3.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::between()     masks data.table::between()
✖ dplyr::filter()      masks stats::filter()
✖ dplyr::first()       masks data.table::first()
✖ lubridate::hour()    masks data.table::hour()
✖ lubridate::isoweek() masks data.table::isoweek()
✖ lubridate::isoyear() masks data.table::isoyear()
✖ dplyr::lag()         masks stats::lag()
✖ dplyr::last()        masks data.table::last()
✖ lubridate::mday()    masks data.table::mday()
✖ lubridate::minute()  masks data.table::minute()
✖ lubridate::month()   masks data.table::month()
✖ lubridate::quarter() masks data.table::quarter()
✖ lubridate::second()  masks data.table::second()
✖ purrr::transpose()   mas

## Load input data

In [6]:
res_adx  <- fread(file.path(out, 'xQTL_all_methods_overlap_with_AD_loci_unified_cs95orColocs_Pval1e5_202604.csv.gz'))
res_adfv <- fread(file.path(out, 'AD_loci_unified_cs95orColocs_Pval1e5_variant_level.csv.gz'))
cols      <- fread(file.path(out, 'columns_metadata.tsv'))[(keep==1)]
colsmtd   <- fread(file.path(out, 'excel_metadata.tsv'))
colorsmtd <- fread(file.path(out, 'pattern_coloring.tsv'))

cat("Loaded:", nrow(res_adx), "rows\n")
cat("Methods:\n"); print(table(res_adx$Method, useNA="ifany"))

Loaded: 2815543 rows
Methods:

       AD_GWAS_finemapping     AD_meta_colocalization 
                      6552                       7825 
    AD_xQTL_colocalization           APOE interaction 
                    142480                       2900 
                     Coloc                         QR 
                     18653                     236849 
                   TWAS/MR                      cTWAS 
                      2689                        118 
        fSuSiE_finemapping           msex interaction 
                     14878                         23 
 multi_context_finemapping     multi_gene_finemapping 
                    106658                      70541 
single_context_finemapping                    sn_sQTL 
                   1501526                     233527 
         trans_finemapping 
                    470324 


## Reproduce xQTL AD loci table

### 1. Summarize + Wide table

In [12]:
# suppress the warning messages
options(warn = -1)

res_adx <- SummarizeTable(res_adx, group.by='context_short')

res_adxub <- WideTable(res_adx, split.by=c('context_broad2','qtl_type'))

res_adxub[, top_variants:=((max_variant_inclusion_probability>=0.1)|cV2F_rank<=5)|
            ((max_variant_inclusion_probability_rank==1|variant_rank_xqtl==1|
               (rank(pval)==1&!is.na(pval)))), by='locus_index']
res_adxubf <- res_adxub[(top_variants)][!is.na(locus_index)][variant_ID!='']


summarizing per context_short the xQTL evidence for each variant

generating columns per context_broad2 and qtl_type

Brain gpQTL

Immune (bulk) eQTL

Brain eQTL

Exc eQTL

Mic eQTL

Oli eQTL

Inh eQTL

Brain pQTL

Ast eQTL

OPC eQTL

Brain mQTL

Brain sQTL

Exc sQTL

Brain p_sQTL

Brain u_sQTL

Inh sQTL

Oli sQTL

Ast sQTL

Brain haQTL

OPC sQTL

Mic sQTL

Ast caQTL

Mic caQTL



### 2. Prepare columns metadata

In [13]:
cols <- PrepColsMtd(cols, colsmtd, res_adxubf)

In [14]:
# main sheet
wb <- CreateExcelFormat(res_adxubf, columns_mtd=cols, colors=colorsmtd)

# One sheet per broad context 
for(cont in colsmtd[wildcard=='context_broad2']$r_name){
  message(cont)
  res_adxc   <- res_adx[context_broad2==cont]
  res_adxc   <- SummarizeTable(res_adxc, group.by='qtl_type')
  res_adxcub <- WideTable(res_adxc)
  
  res_adxcub[, top_variants:=((max_variant_inclusion_probability>=0.1)|cV2F_rank<=5)|
               ((max_variant_inclusion_probability_rank==1|variant_rank_xqtl==1|
                  (rank(pval)==1&!is.na(pval)))), by='ADlocus']
  res_adxcubf <- res_adxcub[(top_variants)]
  
  res_adxcubf[, gwas_significance:=ifelse(min_pval<5e-8,'genome wide',
                                    ifelse(min_pval<1e-6,'suggestive','ns'))]
  res_adxcubf[, mlog10pval:=-log10(min_pval)]
  res_adxcubf[, top_confidence:=str_extract(xQTL_effects,'CL[0-9]')]
  
  wb <- CreateExcelFormat(res_adxcubf, columns_mtd=cols, colors=colorsmtd,
                          wb=wb, sheet_name=cont)
}

pvalue_sexFbeta_sexFse_sexFpvalue_sexF_interactionbeta_sexF_interactionse_sexF_interactionpvalue_sexbeta_sexse_sexpvalue_sex_interactionbeta_sex_interactionse_sex_interactionpvalue_genderbeta_genderse_genderpvalue_gender_interactionbeta_gender_interactionse_gender_interactionresourcebetahatsebetahatpnum_CSnum_IVcpipmeta_effse_meta_effmeta_pvalQQ_pvalI2gwas_studymolecular_idmethodis_imputableis_selected_methodrsq_cvpval_cvtwas_ztwas_pvaltypeblockidregion_idsusie_pipmucsfiletrans_genestrans_contextspurityz_scoremax_finemap_pip.Brain.gpQTLmax_finemap_effect.Brain.gpQTLmax_multicontext_pip.Brain.gpQTLmax_multicontext_effect.Brain.gpQTLmax_MR.Brain.gpQTLmax_QR_padj.Brain.gpQTLmax_APOE4_interaction_pval.Brain.gpQTLmax_msex_interaction_pval.Brain.gpQTLmax_APOE4_interaction_pval.Immune (bulk).eQTLmax_APOE4_interaction_pval.Brain.eQTLmax_msex_interaction_pval.Brain.eQTLmax_msex_interaction_pval.Exc.eQTLmax_APOE4_interaction_pval.Mic.eQTLmax_msex_interaction_pval.Mic.eQTLmax_msex_interaction_pva

### 3. Save excel sheets

In [23]:
saveWorkbook(wb, file.path(out, 'unified_AD_loci_xQTL_summary.xlsx'), overwrite=TRUE)
cat("Saved: unified_AD_loci_xQTL_summary.xlsx\n")

Saved: unified_AD_loci_xQTL_summary.xlsx


## Deploy Shiny app

#### **First-time user** 

**Step 1 — Install packages**

```r
install.packages(c("shiny", "DT", "dplyr", "readr", "stringr", "bslib", "rsconnect"))
```

**Step 2 — Connect your shinyapps.io account**

Get your token at: https://www.shinyapps.io/admin/#/tokens

```r
library(rsconnect)

rsconnect::setAccountInfo(
  name   = "YOUR_SHINYAPPS_USERNAME",
  token  = "YOUR_TOKEN",
  secret = "YOUR_SECRET"
)
```

**Step 3 — Deploy**

```r
rsconnect::deployApp(
  appDir  = "/path/to/shiny_app/",
  appName = "AD-xQTL-Explorer",
  account = "YOUR_SHINYAPPS_USERNAME"
)
```

Your app will be live at `https://YOUR_USERNAME.shinyapps.io/AD-xQTL-Explorer/`

### 1. Prepare `data.csv` for the app

In [25]:
shiny_app_dir <- file.path(out, 'shiny_app/')

shiny_dat <- copy(res_adxubf)
shiny_dat[, significance := ifelse(min_pval < 5e-8, 'genome wide',
                             ifelse(min_pval < 1e-6, 'suggestive', 'ns'))]
shiny_dat[, log10pval := -log10(min_pval)]

old_names <- c('gene_name','SNP','gwas_sources',
                'max_variant_inclusion_probability',
                'max_variant_inclusion_probability_method',
                'cV2F','cV2F_rank',
                'variant_inclusion_probability_xqtl','variant_rank_xqtl',
                'twas_z_gene_max','twas_z_gene_max_context',
                'TWAS_signif_gene','MR_signif_gene','cTWAS_signif_gene',
                'distance_from_tss','distance_from_tes',
                'xQTL_effects','gene_ID','context_xqtl','effect_xqtl')
new_names <- c('gene','rsid','gwas_assoc',
                'max_inclusion','max_inclusion_method',
                'cv2f_score','cv2f_rank',
                'xqtl_max_inclusion','variant_rank',
                'max_twas_z','max_twas_ctx',
                'twas_sig','mr_sig','ctwas_sig',
                'dist_tss','dist_tes',
                'ordered_contexts','gene_id','context','effect')
for (i in seq_along(old_names)) {
  if (old_names[i] %in% names(shiny_dat) && !new_names[i] %in% names(shiny_dat))
    setnames(shiny_dat, old_names[i], new_names[i])
}

for_ct <- function(dt, pattern) {
  cols <- grep(pattern, names(dt), value=TRUE)
  if (length(cols) == 0) return(rep(FALSE, nrow(dt)))
  rowSums(!is.na(dt[, .SD, .SDcols=cols])) > 0
}
shiny_dat[, ct_Brain_xQTL       := for_ct(shiny_dat, "^incl_prob\\.Brain\\.")]
shiny_dat[, ct_Exc_xQTL         := for_ct(shiny_dat, "^incl_prob\\.Exc\\.")]
shiny_dat[, ct_Inh_xQTL         := for_ct(shiny_dat, "^incl_prob\\.Inh\\.")]
shiny_dat[, ct_Oli_xQTL         := for_ct(shiny_dat, "^incl_prob\\.Oli\\.")]
shiny_dat[, ct_OPC_xQTL         := for_ct(shiny_dat, "^incl_prob\\.OPC\\.")]
shiny_dat[, ct_Ast_xQTL         := for_ct(shiny_dat, "^incl_prob\\.Ast\\.")]
shiny_dat[, ct_Microglia_xQTL   := for_ct(shiny_dat, "^incl_prob\\.Mic\\.")]
shiny_dat[, ct_Bulk_Immune_xQTL := for_ct(shiny_dat, "^incl_prob\\.Immune")]

fwrite(shiny_dat, file.path(shiny_app_dir, 'data.csv'))
cat("Saved data.csv\n")


Saved data.csv


### 2. Deploy shiny app

In [26]:
rsconnect::deployApp(
  appDir  = file.path(out,"shiny_app/"),
  appName = "xQTL-AD-loci-Explore",
  account = "jenny-empawi"    # change to your shiny username
)

── Preparing for deployment ────────────────────────────────────────────────────

✔ Re-deploying "xQTL-AD-loci-Explore" using "server: shinyapps.io / username: jenny-empawi"

ℹ Looking up content with id "17249044"...

✔ Found content <https://jenny-empawi.shinyapps.io/xQTL-AD-loci-Explore/>

ℹ Bundling 3 files: DEPLOY.md, app.R, and data.csv

ℹ Capturing R dependencies

✔ Found 62 dependencies

✔ Created bundle of size: 978,453b

ℹ Uploading bundle...

✔ Uploaded bundle with id 11925442

── Deploying to server ─────────────────────────────────────────────────────────



Waiting for task: 1685630387
  building: Processing bundle: 11925442
  building: Building image: 14846335
  building: Installing system dependencies
  building: Fetching packages
  building: Installing packages
  building: Installing files
  building: Pushing image: 14846335
  deploying: Starting instances
  rollforward: Activating new instances
  terminating: Stopping old instances


── Deployment complete ─────────────────────────────────────────────────────────

✔ Successfully deployed to <https://jenny-empawi.shinyapps.io/xQTL-AD-loci-Explore/>

